# Exercise 3: Mining with a shared script

*Corpora at Scale · version 0.1.0*

> Mine Kazakh–Kyrgyz sentence pairs from comparable Wikipedia articles in two ways: character overlap after conversion to the Common Turkic Alphabet, and LaBSE similarity. Check 100 pairs from each method by hand. Which is more precise, and where does each fail?

**Setup.** `pip install "turkicnlp==0.5.0" pyhfst fasttext-wheel huggingface_hub requests`. Open the notebook from the `exercises/` folder: the first code cell imports the chapter's audit functions from `../code/remaining/corpus.py`. GlotLID (`model_v3.bin`, about 1.6 GB) is downloaded from Hugging Face once; set `GLOTLID_PATH` to use a local copy. The Apertium analysers come with TurkicNLP and are read with the pure-Python `pyhfst`. The text comes from the X-WMT test sets on GitHub, so internet access is needed.

Mining from comparable Wikipedia articles needs a hand check, because nobody knows which sentences are translations. A first, cheaper step is to run the same methods on a parallel test set, where the answer is known: the 500-sentence Kazakh–Kyrgyz X-WMT test set, with every Kyrgyz sentence as a candidate for every Kazakh one. This measures how well each score finds a translation among 499 distractors from the same domain. It is easier than real mining, where most candidates have no translation at all, so the precision here is an upper bound.

In [ ]:
import os, re, sys, unicodedata
from collections import Counter
import requests

sys.path.insert(0, os.path.abspath("../code/remaining"))      # the book's corpus.py
from corpus import audit, script_shares, fst_coverage

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """Non-empty lines of one side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return [line.strip() for line in r.text.splitlines() if line.strip()]

def glotlid_model():
    """Unrestricted GlotLID v3 (fastText). Uses GLOTLID_PATH if set, otherwise downloads the model."""
    import fasttext
    path = os.environ.get("GLOTLID_PATH")
    if not path:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
    return fasttext.load_model(path)

def lid_labels(model, text, k=1):
    """Labels, best first. A list input avoids a NumPy 2 incompatibility in fasttext's single-string path."""
    return model.predict([text.replace("\n", " ")], k=k)[0][0]

def load_analyser(lang, script="Cyrl"):
    """Apertium analyser from TurkicNLP, read with pyhfst. lookup(word) -> [(reading, weight), ...]."""
    import pyhfst, turkicnlp
    from turkicnlp.resources.registry import ModelRegistry
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
    fst = pyhfst.HfstInputStream(str(path)).read()
    def lookup(word):
        readings = fst.lookup(word) or fst.lookup(word.lower())
        return [(reading, float(weight)) for reading, weight in readings]
    return lookup

WORD = re.compile(r"[^\W\d_]+")
def words(text):
    return WORD.findall(unicodedata.normalize("NFC", text))

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

kk, ky = xwmt_side("kk-ky", "kk"), xwmt_side("kk-ky", "ky")
assert len(kk) == len(ky)            # line i of one side translates line i of the other
to_cts_kk = Transliterator("kaz", Script.CYRILLIC, Script.COMMON_TURKIC)
to_cts_ky = Transliterator("kir", Script.CYRILLIC, Script.COMMON_TURKIC)

def grams(s, n=3):
    s = " " + re.sub(r"\s+", " ", s.lower()) + " "
    return Counter(s[i:i + n] for i in range(len(s) - n + 1))

def dice(a, b):
    total = sum(a.values()) + sum(b.values())
    return 2 * sum((a & b).values()) / total if total else 0.0

def mine(src, tgt):
    """For each source sentence: best target index, its score and the margin over the second best."""
    G = [grams(t) for t in tgt]
    out = []
    for i, s in enumerate(src):
        scores = [dice(grams(s), g) for g in G]
        order = sorted(range(len(G)), key=scores.__getitem__, reverse=True)
        out.append((i, order[0], scores[order[0]], scores[order[0]] - scores[order[1]]))
    return out

def report(name, pairs):
    p1 = sum(i == j for i, j, _, _ in pairs) / len(pairs)
    print(f"{name}: precision@1 = {p1:.3f} on {len(pairs)} sentences")
    for t in (0.0, 0.05, 0.10, 0.15):
        kept = [(i, j) for i, j, _, m in pairs if m >= t]
        prec = sum(i == j for i, j in kept) / len(kept) if kept else float("nan")
        print(f"   margin >= {t:.2f}: kept {len(kept):3d}, precision {prec:.3f}, recall {sum(i == j for i, j in kept) / len(pairs):.3f}")

native = mine(kk, ky)
cts = mine([to_cts_kk.transliterate(s) for s in kk], [to_cts_ky.transliterate(s) for s in ky])
report("native Cyrillic", native)
report("CTS", cts)

In [ ]:
wrong = [(i, j, round(s, 3)) for i, j, s, m in cts if i != j][:3]
for i, j, s in wrong:
    print(f"kk[{i}]: {kk[i][:90]}\n  chosen ky[{j}] (score {s}): {ky[j][:90]}\n  gold   ky[{i}]: {ky[i][:90]}\n")

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    labse = SentenceTransformer("sentence-transformers/LaBSE")
    E_kk = labse.encode(kk, normalize_embeddings=True); E_ky = labse.encode(ky, normalize_embeddings=True)
    best = (E_kk @ E_ky.T).argmax(axis=1)
    print("LaBSE precision@1:", round(float(np.mean(best == np.arange(len(kk)))), 3))
except Exception as exc:
    print("LaBSE not available here:", type(exc).__name__, "- install sentence-transformers and allow the model download.")

## Worked solution

Output on 18 October 2026, UTC (character trigrams, Dice coefficient; margin = best score minus second-best score):

```
native Cyrillic: precision@1 = 0.840 on 500 sentences
   margin >= 0.00: kept 500, precision 0.840, recall 0.840
   margin >= 0.05: kept 335, precision 0.955, recall 0.640
   margin >= 0.10: kept 225, precision 0.956, recall 0.430
   margin >= 0.15: kept 122, precision 0.951, recall 0.232
CTS: precision@1 = 0.860 on 500 sentences
   margin >= 0.00: kept 500, precision 0.860, recall 0.860
   margin >= 0.05: kept 373, precision 0.960, recall 0.716
   margin >= 0.10: kept 257, precision 0.957, recall 0.492
   margin >= 0.15: kept 158, precision 0.956, recall 0.302
```

```
kk[0]: Мақсат - лаңкестерге су арқылы ену , деп қосты ақпараттық агенттік .
  chosen ky[143] (score 0.144): Жергиликтүү маалымат агенттиктеринин билдиришинче , Милуд Брахиминин жактоочуларынын бири
  gold   ky[0]: Индиянын чалгындоо бюросу билдиргендей , террористтер деңизге кирүү ыкмаларын үйрөтүп жаты

kk[3]: Пәкістандағы сәтсіз оқиғаға ұшақтарды қолданып , қару-жарақ пен оқ-дәрілерді орналастыру -
  chosen ky[411] (score 0.201): Комментарийлер Пакистандын массалык маалымат каражаттарында резонанс жаратып , крикет дүйн
  gold   ky[3]: Пакистан террорчулары үчүн жаңы стратегиянын эң сонун мисалы - чек арадан курал-жарак жана

kk[8]: Құрлықпен байланыстыру мүмкін емес болғандықтан , Пәкістан лаңкестері әуе және су арқылы к
  chosen ky[398] (score 0.152): Андан бери ал сегиз жолу талаага чыгып , ар бир жолу өзүнүн оюну менен таасирлентти , анын
  gold   ky[8]: Жерге кирүү жабылды : Пакистандык террористтер Индияга деңиз жана аба аркылуу кирүүнү план
```

```
LaBSE not available here: ModuleNotFoundError - install sentence-transformers and allow the model download.
```

Among 500 candidates, the best character-trigram match is the translation for 84.0% of the Kazakh sentences in Cyrillic and for 86.0% after both sides are converted to CTS, a difference of 10 sentences on this set. Requiring a margin of at least 0.05 over the second-best candidate raises precision to about 96% for both representations, and CTS keeps more pairs at that precision (373 against 335). On this test set, conversion therefore mainly helps recall. The errors pick sentences that share a name or a topic word with the source (*агенттік* / *агенттиктеринин*, *Пәкістан* / *Пакистандын*). Some gold pairs are also free translations: the two versions of sentence 0 overlap only partly in content. This is why the exercise asks for a hand check instead of trusting any alignment.

LaBSE was not available in the environment used for the worked solution; with `sentence-transformers` installed and the model downloadable, the last cell prints its precision@1 in the same 500 × 500 setting. On comparable Wikipedia articles, expect lower precision for both methods: most candidate sentences there have no translation at all, and character overlap rewards shared names regardless of meaning.

## Your turn — extension

1. Collect Kazakh and Kyrgyz Wikipedia articles on the same topics (linked through Wikidata), split them into sentences, and mine with both methods. Check 100 proposed pairs per method by hand, labelling each as translation, partial overlap, or only on the same topic.
2. Report precision at the same number of proposed pairs for both methods, so the comparison is fair.
3. Try word-level overlap (shared CTS word types) instead of character trigrams and see where it fails.